# 02 — Where Monge Breaks

Meet the obstruction that defeats Monge's deterministic map: when one pile of mass must *split* to fill another, no function can describe the move.

**Prerequisites:** `01_the_monge_problem`.

**What you'll be able to do**
- Construct a source and target where *no* deterministic Monge map exists.
- Prove it by exhaustion — check every possible map and watch them all fail the push-forward.
- Name the missing capability, mass splitting, that the next notebook restores.

In the last notebook a Monge map slid neatly into place because the masses matched one-to-one. That was a kindness of the setup, not a law. Here we build the honest case — two source piles, three target piles — and find that Monge's beautiful map has nowhere to go. Naming this failure precisely is what earns us Kantorovich's fix.

In [ ]:
import itertools

import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz

np.random.seed(42)
viz.use_course_style()

## 1. A target that cannot be matched one-to-one

Take **two** source atoms, at positions $0$ and $4$, each carrying mass $\tfrac{1}{2}$. Take **three** target atoms, at positions $1$, $2$, $3$, each needing mass $\tfrac{1}{3}$. The totals agree — one unit on each side — so a transport *plan* of some kind must exist. But a Monge **map** is a function: each source atom sends *all* of its mass to a single target. With only two sources, a map can touch at most **two** of the three targets, and the third is left empty.

In [ ]:
x_source = np.array([0.0, 4.0])
x_target = np.array([1.0, 2.0, 3.0])
a = np.array([0.5, 0.5])
b = np.array([1 / 3, 1 / 3, 1 / 3])

print(f"total source mass = {a.sum():.3f},   total target mass = {b.sum():.3f}")

viz.plot_distributions(a, b, source_positions=x_source, target_positions=x_target)
plt.show()

**Read the figure.** Two tall periwinkle bars (mass $\tfrac{1}{2}$ each) face three shorter amber bars (mass $\tfrac{1}{3}$ each). The mass budgets balance, so nothing is *globally* wrong — the difficulty is local: a function must commit each source to one destination, and two commitments cannot cover three needs.

## 2. Every deterministic map fails — by exhaustion

We need not argue in the abstract. There are only $3^2 = 9$ functions from two source atoms to three targets. Let's enumerate every one of them and check the push-forward condition $T_\#\mu = \nu$ — that the delivered mass equals $b$ on every target.

In [ ]:
def delivered_mass(assignment):
    """Mass landing on each target under a deterministic map (no splitting allowed)."""
    delivered = np.zeros(len(b))
    for source_idx, target_idx in enumerate(assignment):
        delivered[target_idx] += a[source_idx]
    return delivered

feasible = 0
for assignment in itertools.product(range(3), repeat=2):   # all 9 maps T: {0, 1} -> {0, 1, 2}
    if np.allclose(delivered_mass(assignment), b):
        feasible += 1
        print(f"  feasible map: {assignment}")

print(f"feasible Monge maps (out of 9): {feasible}")

# The 'best effort' nearest-target map, to see the failure concretely.
nearest = np.array([np.argmin(np.abs(x_target - xs)) for xs in x_source])
print(f"\nnearest-target map {nearest.tolist()} delivers {np.round(delivered_mass(nearest), 3).tolist()}")
print(f"target needs              {np.round(b, 3).tolist()}")

In [ ]:
plan_nearest = np.zeros((len(a), len(b)))
for source_idx, target_idx in enumerate(nearest):
    plan_nearest[source_idx, target_idx] = a[source_idx]

viz.plot_transport_arrows(
    a, b, plan_nearest,
    source_positions=x_source, target_positions=x_target,
    title="Best-effort Monge map: the middle target is left empty",
)
plt.show()

**Read the output and figure.** Out of all nine candidate maps, **zero** satisfy the push-forward. The nearest-target attempt sends both sources to the outer targets and leaves the middle one (at $x = 2$) with no mass — you can see the gap in the flow view, where no line reaches the centre. And the targets that *do* receive mass get $\tfrac{1}{2}$, never the required $\tfrac{1}{3}$: a whole source pile is too much, but a function cannot hand over part of a pile. Monge is infeasible here, and no cleverness rescues it.

## 3. The missing capability: splitting mass

The obstruction has a one-word name: **splitting**. To fill three targets from two sources, at least one source must send part of its mass one way and part another — a *fractional* assignment that a function cannot represent. Kantorovich's move was to stop insisting on a function and allow exactly this: a plan in which a single source spreads its mass across several destinations. That is the relaxation we build next, and it always has a solution.

## Your turn

1. **Close the gap.** Change the target to two atoms with masses `[0.5, 0.5]` at positions you choose, and re-run the enumeration. Does a feasible Monge map appear now? What made the difference?
2. **Mass mismatch alone.** Keep two sources and two targets, but set masses `a = [0.5, 0.5]` and `b = [0.7, 0.3]`. Can any of the four maps deliver `b`? What does this say about *when* a one-to-one map can work?
3. **Count reachable targets.** With $n$ source atoms and a deterministic map, how many distinct targets can receive mass, at most? Use that bound to predict — without enumerating — when Monge is hopeless.

## What you built

- You constructed a source and target where no deterministic Monge map exists.
- You proved it by exhaustion — all nine candidate maps fail the push-forward $T_\#\mu = \nu$.
- You named the missing capability, mass splitting, that a function cannot express.

Finding the wall is real progress: you now know *exactly* what Monge lacks, which is the precise thing Kantorovich adds. Naming a limitation is the first step past it.

## What's next

In `03_the_kantorovich_lp` we relax the map into a **coupling** — a joint plan that lets mass split freely — and recast optimal transport as a linear program that is *always* solvable. That empty middle target will fill in.

## References

- L. V. Kantorovich, "On the translocation of masses", *Doklady Akademii Nauk SSSR* 37, 199–201 (1942).
- C. Villani, *Topics in Optimal Transportation*, AMS (2003), ch. 1.
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 2. DOI:10.1561/2200000073

**Previous:** `notebooks/03_ClassicalOptimalTransport/01_the_monge_problem.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/03_the_kantorovich_lp.ipynb`